In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.utils import to_categorical

In [2]:
text = """
the quick brown fox jumps over the lazy dog
the dog barked at the fox
the fox ran into the forest
"""

text = text.lower().strip()
print("Text length:", len(text))
print("Sample:", text[:60])

Text length: 97
Sample: the quick brown fox jumps over the lazy dog
the dog barked a


In [3]:

chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

vocab_size = len(chars)
print("Vocab size:", vocab_size)
print("Characters:", chars)

Vocab size: 28
Characters: ['\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:

seq_length = 20

X, y = [], []
encoded = [char2idx[c] for c in text]

for i in range(len(encoded) - seq_length):
    X.append(encoded[i : i + seq_length])
    y.append(encoded[i + seq_length])

X = np.array(X)
y = to_categorical(y, num_classes=vocab_size)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (77, 20)
y shape: (77, 28)


In [5]:

model = Sequential([
    Embedding(vocab_size, 32, input_length=seq_length),
    LSTM(128, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:

history = model.fit(X, y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.0909 - loss: 3.3284
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1948 - loss: 3.3076
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 3.2747
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 3.1956
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1818 - loss: 3.0211
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 2.9703
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 2.9008
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 2.8970
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1818 - loss: 2.8630
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1948 - loss: 2.8474
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1818 - loss: 2.8663
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.1818 - lo

In [7]:

def generate_text(seed, num_chars=200, temperature=1.0):
    seed = seed.lower()
    generated = seed

    for _ in range(num_chars):
        # Encode and pad seed
        encoded = [char2idx.get(c, 0) for c in seed[-seq_length:]]
        encoded = np.array(encoded).reshape(1, -1)

        preds = model.predict(encoded, verbose=0)[0]

        preds = np.log(preds + 1e-10) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))

        next_idx = np.random.choice(len(preds), p=preds)
        next_char = idx2char[next_idx]

        generated += next_char
        seed += next_char

    return generated

In [8]:

seed_text = "the quick"
output = generate_text(seed_text, num_chars=150, temperature=0.8)
print("Generated text:\n")
print(output)

Generated text:

the quickqibkbjkjkkjkuuzuuumppdbsysyupykbddboggdd    d
athe t   orrette   aargddtoe  aarate   atxe  aoxthehe ogo
eth do  agndd
the  a  thee  a ththedooorg thhe
